# Logistic Regression — Yeo-Johnson + RobustScaler

Standalone notebook training and evaluating Logistic Regression with Yeo-Johnson + RobustScaler preprocessing.

**Input:** `datasets/phishing_url_features.csv`  
**Output:** Best model saved to `models/` if it beats XGBoost baseline  
**Primary metric:** Recall (minimise false negatives — undetected phishing)

**Differences from `03_modeling.ipynb`:**
- Preprocessing: Yeo-Johnson + RobustScaler (instead of log1p + StandardScaler)
- Single model: Logistic Regression only
- No stacking, no feature selection

## SECTION 1 — Imports

In [8]:
from pathlib import Path

import joblib
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    precision_recall_curve, ConfusionMatrixDisplay,
    average_precision_score,
)

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)
print('All libraries imported.')

All libraries imported.


## SECTION 2 — Data Loading

In [9]:
data_path = Path('./datasets/phishing_url_features.csv')
if not data_path.exists():
    raise FileNotFoundError(f'Feature CSV not found at {data_path}. Run 02_feature_engineering.ipynb first.')

df = pd.read_csv(data_path)
print(f'Loaded: {len(df):,} rows x {df.shape[1]} columns')
print(f'Phishing: {(df["label"]==1).sum():,} | Legitimate: {(df["label"]==0).sum():,}')
df.head()

Loaded: 5,183,050 rows x 29 columns
Phishing: 2,301,110 | Legitimate: 2,881,940


,url,URLLength,Domain,DomainLength,TLD,TLDLength,NoOfSubDomain,SubdomainLength,NoOfLettersInURL,NoOfDigitsInURL,LetterRatioInURL,DigitRatioInURL,NoOfSpecialCharsInURL,NoOfDots,NoOfDashInURL,PathLength,PathDepth,IsHTTPS,HasIPAddress,HasHyphenInDomain,HasSuspiciousKeyword,DomainHasNumber,DomainEntropy,HasAtSymbol,HasDoubleSlashInPath,URLHasPort,HasHexEncoding,NoOfQueryParams,label
0,https://blockchaoin.info/#/,27,blockchaoin,11,info,4,0,0,20,0,0.740741,0.000000,7,1,0,1,0,1,0,0,0,0,3.095795,0,0,0,0,0,1
1,https://www.insects.org/ced4/crush_freaks.html,46,insects,7,org,3,0,0,36,1,0.782609,0.021739,9,3,0,23,2,1,0,0,0,0,2.521641,0,0,0,0,0,0
2,https://www.elpasotimes.com/story/opinion/edit...,118,elpasotimes,11,com,3,0,0,83,16,0.703390,0.135593,19,2,5,91,8,1,0,0,0,0,3.095795,0,0,0,0,0,0
3,http://direct-certs.bankofamerica.com.techdbas...,147,techdbaseurl46,14,cn,2,3,30,71,63,0.482993,0.428571,13,5,1,22,2,0,0,0,1,1,3.664498,0,0,0,0,1,1
4,https://gotham-magazine.com/lalique-unveils-ep...,53,gotham-magazine,15,com,3,0,0,44,0,0.830189,0.000000,9,1,4,26,1,1,0,1,0,0,3.323231,0,0,0,0,0,0


## SECTION 3 — Feature Selection

In [10]:
# 21 features after correlation-based pruning (see 02_feature_engineering.ipynb Section 6.7)
# Dropped: SubdomainLength, NoOfDigitsInURL, NoOfLettersInURL, LetterRatioInURL
FEATURES = [
    # Length-based
    'URLLength', 'DomainLength', 'TLDLength', 'PathLength', 'PathDepth',
    # Subdomain
    'NoOfSubDomain',
    # Character composition
    'DigitRatioInURL', 'NoOfSpecialCharsInURL', 'NoOfDots', 'NoOfDashInURL',
    # Binary structural indicators
    'IsHTTPS', 'HasIPAddress', 'HasHyphenInDomain', 'HasSuspiciousKeyword',
    'DomainHasNumber', 'HasAtSymbol', 'HasDoubleSlashInPath',
    'URLHasPort', 'HasHexEncoding',
    # Complexity
    'DomainEntropy', 'NoOfQueryParams',
]  # 21 features

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    raise ValueError(f'Missing features: {missing}')

X = df[FEATURES]
y = df['label']
print(f'Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features')

Feature matrix: 5,183,050 rows x 21 features


## SECTION 4 — Train/Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train phishing rate: {y_train.mean():.4f} | Test phishing rate: {y_test.mean():.4f}')

Train: 4,146,440 | Test: 1,036,610
Train phishing rate: 0.4440 | Test phishing rate: 0.4440


## SECTION 5 — Preprocessing: Yeo-Johnson + RobustScaler

Yeo-Johnson handles zeros and negatives natively and outperformed log1p on this dataset.
RobustScaler uses median and IQR — more resistant to outliers than StandardScaler.

In [12]:
# Pipeline: Yeo-Johnson → RobustScaler → LogisticRegression
# All steps are inside the Pipeline so fit/transform is handled automatically.
# The pipeline takes raw X_train/X_test directly.

# CV setup
skf = StratifiedShuffleSplit(n_splits=3, test_size=0.2, random_state=42)

print('Preprocessing pipeline: Yeo-Johnson → RobustScaler → LogisticRegression')
print(f'CV: 3-fold StratifiedShuffleSplit')

Preprocessing pipeline: Yeo-Johnson → RobustScaler → LogisticRegression
CV: 3-fold StratifiedShuffleSplit


## SECTION 6 — Baseline

In [13]:
baseline_pipe = Pipeline([
    ('yj',     PowerTransformer(method='yeo-johnson')),
    ('scaler', RobustScaler()),
    ('clf',    LogisticRegression(C=1.0, solver='saga', class_weight='balanced', max_iter=5000, random_state=42)),
])

baseline_pipe.fit(X_train, y_train)
y_pred_base = baseline_pipe.predict(X_test)
y_prob_base = baseline_pipe.predict_proba(X_test)[:, 1]

baseline_df = pd.DataFrame([{
    'Model':     'LogReg (YJ+Robust, default)',
    'Accuracy':  round(accuracy_score(y_test, y_pred_base), 4),
    'Recall':    round(recall_score(y_test, y_pred_base), 4),
    'Precision': round(precision_score(y_test, y_pred_base), 4),
    'F1':        round(f1_score(y_test, y_pred_base), 4),
    'ROC-AUC':   round(roc_auc_score(y_test, y_prob_base), 4),
}]).set_index('Model')

print('Baseline (default threshold=0.5):')
display(baseline_df)

Baseline (default threshold=0.5):


,Accuracy,Recall,Precision,F1,ROC-AUC
Model,,,,,
"LogReg (YJ+Robust, default)",0.8969,0.9142,0.862,0.8873,0.9676


## SECTION 7 — Optuna Hyperparameter Tuning

100 trials, TPE sampler, maximise recall via 3-fold CV.

In [ ]:
def objective_logreg(trial):
    params = {
        'C':      trial.suggest_float('C', 1e-3, 100, log=True),
        'solver': trial.suggest_categorical('solver', ['saga', 'lbfgs']),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X_train, y_train):
        pipe = Pipeline([
            ('yj',     PowerTransformer(method='yeo-johnson')),
            ('scaler', RobustScaler()),
            ('clf',    LogisticRegression(
                **params, class_weight='balanced', max_iter=1000, random_state=42)),
        ])
        pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        scores.append(recall_score(
            y_train.iloc[val_idx],
            pipe.predict(X_train.iloc[val_idx])
        ))
    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_logreg, n_trials=100)

best_params = {k: v for k, v in study.best_params.items()}
print(f'Best recall (CV): {study.best_value:.4f}')
print(f'Best params: {best_params}')

In [ ]:
# Optuna optimization history
fig, ax = plt.subplots(figsize=(10, 5))
trials = [t.value for t in study.trials if t.value is not None]
best   = [max(trials[:i+1]) for i in range(len(trials))]
ax.plot(trials, alpha=0.3, color='#239DD9', linewidth=0.8, label='Trial recall')
ax.plot(best,   color='#239DD9', linewidth=2, label='Best so far')
ax.axhline(y=study.best_value, color='grey', linestyle='--', linewidth=1)
ax.set_xlabel('Trial')
ax.set_ylabel('Recall (CV)')
ax.set_title(f'Optuna Optimization History — LogReg (YJ+Robust)\nBest: {study.best_value:.4f}', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## SECTION 8 — Final Model Training & Evaluation

In [ ]:
best_model = Pipeline([
    ('yj',     PowerTransformer(method='yeo-johnson')),
    ('scaler', RobustScaler()),
    ('clf',    LogisticRegression(
        **best_params, class_weight='balanced', max_iter=5000, random_state=42)),
])

best_model.fit(X_train, y_train)
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

eval_df = pd.DataFrame([{
    'Model':     'LogReg (YJ+Robust, tuned)',
    'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
    'Recall':    round(recall_score(y_test, y_pred), 4),
    'Precision': round(precision_score(y_test, y_pred), 4),
    'F1':        round(f1_score(y_test, y_pred), 4),
    'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
}]).set_index('Model')

print('Tuned model (default threshold=0.5):')
display(eval_df.style.highlight_max(axis=0, color='#239DD9'))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Phishing'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix (threshold=0.5)', fontweight='bold')
plt.tight_layout()
plt.show()

## SECTION 9 — Threshold Optimisation

Find lowest threshold where recall ≥ 0.95, maximising F1.

In [ ]:
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, y_prob)

opt_threshold = 0.5
opt_f1        = 0.0
for p, r, t in zip(prec_arr, rec_arr, thresh_arr):
    if r >= 0.95:
        f1 = 2 * p * r / (p + r + 1e-9)
        if f1 > opt_f1:
            opt_f1        = f1
            opt_threshold = t

y_pred_opt = (y_prob >= opt_threshold).astype(int)

thresh_df = pd.DataFrame([{
    'Model':         'LogReg (YJ+Robust)',
    'opt_threshold': round(float(opt_threshold), 4),
    'recall':        round(recall_score(y_test, y_pred_opt), 4),
    'precision':     round(precision_score(y_test, y_pred_opt), 4),
    'f1':            round(f1_score(y_test, y_pred_opt), 4),
    'roc_auc':       round(roc_auc_score(y_test, y_prob), 4),
    'strategy':      'recall>=0.95, max F1',
}]).set_index('Model')

print('Threshold optimisation result:')
display(thresh_df)

In [ ]:
# PR curve
fig, ax = plt.subplots(figsize=(9, 6))
ap = average_precision_score(y_test, y_prob)
ax.plot(rec_arr, prec_arr, color='#239DD9', linewidth=2, label=f'LogReg YJ+Robust (AP={ap:.4f})')
ax.axvline(x=0.95, color='#FF911E', linestyle='--', linewidth=1.2, label='Recall = 0.95 target')
ax.scatter([thresh_df['recall'].values[0]], [thresh_df['precision'].values[0]],
           color='#FF911E', zorder=5, s=80, label=f'Opt threshold ({opt_threshold:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — LogReg (YJ+Robust)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## SECTION 10 — SHAP Explanation Demo

In [ ]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

# Transform X_test through YJ + RobustScaler for SHAP
yj_step     = best_model.named_steps['yj']
scaler_step = best_model.named_steps['scaler']
clf_step    = best_model.named_steps['clf']

SHAP_SAMPLE = min(5_000, len(X_test))
shap_idx    = np.random.RandomState(42).choice(len(X_test), SHAP_SAMPLE, replace=False)
X_shap_raw  = X_test.iloc[shap_idx]
X_shap_transformed = pd.DataFrame(
    scaler_step.transform(yj_step.transform(X_shap_raw)),
    columns=FEATURES
)

bg_transformed = pd.DataFrame(
    scaler_step.transform(yj_step.transform(X_train.sample(n=100, random_state=42))),
    columns=FEATURES
)

explainer  = shap.LinearExplainer(clf_step, bg_transformed)
shap_values = explainer.shap_values(X_shap_transformed)
shap_arr   = shap_values[1] if isinstance(shap_values, list) else shap_values

shap.summary_plot(shap_arr, X_shap_transformed, feature_names=FEATURES, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — LogReg (YJ+Robust)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

shap.summary_plot(shap_arr, X_shap_transformed, feature_names=FEATURES, show=False)
plt.title('SHAP Beeswarm — LogReg (YJ+Robust)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## SECTION 11 — Comparison vs XGBoost Baseline

XGBoost results from `03_modeling.ipynb` (log1p + StandardScaler, Optuna-tuned, threshold-optimised).

In [ ]:
# XGBoost results hardcoded from 03_modeling.ipynb
xgb_results = {
    'Model':         'XGBoost (log1p+Standard, 03_modeling)',
    'opt_threshold': 0.5095,
    'recall':        0.9556,
    'precision':     0.9409,
    'f1':            0.9482,
    'roc_auc':       0.9924,
}

logreg_results = {
    'Model':         'LogReg (YJ+Robust, this notebook)',
    'opt_threshold': float(thresh_df['opt_threshold'].values[0]),
    'recall':        float(thresh_df['recall'].values[0]),
    'precision':     float(thresh_df['precision'].values[0]),
    'f1':            float(thresh_df['f1'].values[0]),
    'roc_auc':       float(thresh_df['roc_auc'].values[0]),
}

comparison_df = pd.DataFrame([xgb_results, logreg_results]).set_index('Model')
print('Final comparison:')
display(comparison_df.style.highlight_max(subset=['recall','precision','f1','roc_auc'], axis=0, color='#239DD9'))

logreg_beats_xgb = logreg_results['recall'] > xgb_results['recall']
print(f'\nLogReg (YJ+Robust) beats XGBoost on recall: {logreg_beats_xgb}')

## SECTION 12 — Model Persistence

Saves the model bundle only if LogReg (YJ+Robust) beats XGBoost on recall.

In [ ]:
if logreg_beats_xgb:
    # Refit on full training data
    final_model = Pipeline([
        ('yj',     PowerTransformer(method='yeo-johnson')),
        ('scaler', RobustScaler()),
        ('clf',    LogisticRegression(
            **best_params, class_weight='balanced', max_iter=5000, random_state=42)),
    ])
    final_model.fit(X_train, y_train)

    out_dir = Path('models')
    out_dir.mkdir(exist_ok=True)
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    model_path = out_dir / f'deploy_LogReg_YJ_Robust_{ts}.pkl'

    joblib.dump({
        'model':           final_model,
        'model_name':      'LogisticRegression_YJ_Robust',
        'features':        FEATURES,
        'threshold':       float(opt_threshold),
        'skewed_features': FEATURES,  # all features passed through YJ
    }, model_path)

    print(f'Model saved to: {model_path}')
else:
    print('LogReg (YJ+Robust) did not beat XGBoost on recall. No bundle saved.')
    print(f'  LogReg recall:  {logreg_results["recall"]:.4f}')
    print(f'  XGBoost recall: {xgb_results["recall"]:.4f}')